# HỆ THỐNG AI BANKING RAG ASSISTANT (VIETCOMBANK)
## BÁO CÁO KẾT QUẢ TRIỂN KHAI: DUAL-INDEXING & 5-STEP RAG PIPELINE

---

### Tổng quan Kiến trúc Kỹ thuật 5 Phân hệ:

```
[Khách hàng đặt câu hỏi]
          │
          ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ BƯỚC 1: QUERY PARSER                                                        │
│ - Chuẩn hóa từ viết tắt: eKYC, TKTT, Smart OTP, VCB Digibank, KHCN, CCCD... │
│ - Trích xuất từ khóa chuyên ngành (Keywords) & Thực thể pháp lý             │
└──────────────────────┬──────────────────────────────────────────────────────┘
                       │ (Normalized Query & Keywords)
                       ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ BƯỚC 2: HYBRID RETRIEVAL (DUAL-INDEXING & RRF k=60)                         │
│ ┌───────────────────────────────────────┐ ┌───────────────────────────────┐ │
│ │ Dense Vector Store (Cosine HNSW 1024d)│ │ Sparse Keyword Search (BM25)  │ │
│ └───────────────────┬───────────────────┘ └───────────────┬───────────────┘ │
│                     └───────────────┬─────────────────────┘                 │
│                                     ▼                                       │
│          Reciprocal Rank Fusion (RRF k=60) ──► Trích xuất TOP-5 Ứng viên    │
└─────────────────────────────────────┬───────────────────────────────────────┘
                                      │ (Top-5 Candidates)
                                      ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ BƯỚC 3: RERANKING                                                           │
│ - So khớp câu hỏi với Câu hỏi chuẩn (Canonical Questions Matching)          │
│ - Tính toán Độ trùng lặp từ vựng (Lexical Overlap & Keyword Containment)    │
│ - Tinh lọc thứ hạng ──► Trích xuất TOP-3 Chunks chính xác nhất              │
└─────────────────────────────────────┬───────────────────────────────────────┘
                                      │ (Top-3 Chunks)
                                      ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ BƯỚC 4: CONTEXT SYNTHESIZER                                                 │
│ - Đóng gói ngữ cảnh Markdown có cấu trúc phân cấp (Breadcrumbs)             │
│ - Gắn thẻ căn cứ pháp lý chính thức: [Điều X, Khoản Y]                      │
│ - Kiểm soát ngân sách token (Token Budget Control <= 2000 tokens)           │
└─────────────────────────────────────┬───────────────────────────────────────┘
                                      │ (Synthesized Markdown Context)
                                      ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ BƯỚC 5: LLM SERVING & BANKING SYSTEM GUARDRAILS                             │
│ - Áp dụng 4 Guardrails ngân hàng: Zero-Hallucination, Dẫn nguồn pháp lý,   │
│   Bảo mật OTP/PIN/CVV, Minh bạch giới hạn thông tin                         │
│ - Mô hình phục vụ: DeepSeek-V4-Flash / Google Gemini                        │
└─────────────────────────────────────┬───────────────────────────────────────┘
                                      │
                                      ▼
                      [CÂU TRẢ LỜI TƯ VẤN CHUẨN XÁC]
```


In [1]:
import os
import sys
from pathlib import Path

# Thêm thư mục gốc dự án vào sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
from dotenv import load_dotenv

# Tải cấu hình môi trường từ thư mục gốc
load_dotenv(ROOT_DIR / ".env")

# Nạp các phân hệ cốt lõi của RAG Banking
from rag_banking.config import Config
from rag_banking.chunking.breadcrumbs_chunker import BreadcrumbsChunker
from rag_banking.embeddings.embedding_model import EmbeddingModel
from rag_banking.vector_db.cosine_hnsw_store import CosineHNSWVectorStore
from rag_banking.keyword_search.bm25_search import BM25Search
from rag_banking.query_parser.banking_query_parser import BankingQueryParser
from rag_banking.retrieval.hybrid_retriever import HybridRetriever
from rag_banking.reranking.banking_reranker import BankingReranker
from rag_banking.synthesizer.context_synthesizer import ContextSynthesizer
from rag_banking.generator.rag_generator import RAGGenerator, BANKING_SYSTEM_GUARDRAILS
from rag_banking.pipeline import BankingRAGPipeline

print("Đã nạp thành công toàn bộ thư viện và mô-đun của Hệ thống AI Banking RAG!")


Đã nạp thành công toàn bộ thư viện và mô-đun của Hệ thống AI Banking RAG!


D:\Desktop\GitHub\AI Banking Asistant\.venv\Lib\site-packages\langsmith\_internal\_beta_decorator.py:70: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  if asyncio.iscoroutinefunction(func):


## PHA INGESTION & DUAL-INDEXING (LẬP CHỈ MỤC KÉP SONG SONG)

### 1. Phân đoạn bảo toàn Breadcrumbs:
Mỗi tài liệu văn bản quy phạm, biểu phí và hướng dẫn được phân đoạn giữ nguyên đường dẫn phân cấp:
$$\text{Breadcrumbs} = \text{Tên tài liệu} > \text{Chương/Mục} > \text{Điều X} > \text{Khoản Y / Câu hỏi}$$
Nội dung chunk luôn có tiền tố: `[Breadcrumbs: ...]\n[Pháp lý: [Điều X, Khoản Y]]\nNội dung`.

### 2. Chỉ mục kép song song (Dual-Indexing):
* **Sparse Index**: Giải thuật **Okapi BM25** với tham số $k_1 = 1.5, b = 0.75$, tính Document Frequency và Inverse Document Frequency (IDF) chuẩn xác.
* **Dense Vector Store**: Không gian vector **1024 chiều** với khoảng cách **Cosine Distance** và cấu trúc đồ thị **HNSW** (Hierarchical Navigable Small World, $M=16, ef_{construct}=100$).


In [2]:
# Khởi tạo Pipeline hoàn chỉnh
pipeline = BankingRAGPipeline()

# Danh sách tài liệu ngân hàng thực tế đại diện từ knowledge_base
curated_files = [
    "phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md",
    "phap_ly__dieu_kien_dieu_khoan_mo_va_su_dung_tai_khoan_khcn.md",
    "quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_mo_so_dep.md",
    "faq__ngan_hang_so_vcb_digibank_va_smart_otp.md",
    "ngan_hang_so__quy_chuan_dich_vu_digibank_va_han_muc.md",
    "tai_khoan__bieu_phi_dich_vu_tai_khoan_thanh_toan.md",
    "cho_vay__vay_tieu_dung_co_tai_san_bao_dam.md"
]

all_chunks = []
dataset_dir = Config.RETRIEVAL_DATASET_DIR

for fname in curated_files:
    fpath = dataset_dir / fname
    if fpath.exists():
        text = fpath.read_text(encoding="utf-8", errors="ignore")
        chunks = pipeline.chunker.chunk_text(text, source_name=fname)
        all_chunks.extend(chunks)

print(f"-> Đã trích xuất tổng cộng: {len(all_chunks)} chunks có cấu trúc Breadcrumbs.")

# Lập chỉ mục kép song song:
# 1. Nạp Dense Vector Store (Cosine HNSW 1024-d)
texts = [c["content"] for c in all_chunks]
print(f"-> Đang sinh vector nhúng 1024 chiều (multilingual-e5-large) cho {len(texts)} chunks...")
vectors = pipeline.embedding_model.embed_documents(texts, batch_size=32)
pipeline.vector_store.add_chunks(all_chunks, vectors)
print(f"   [Dense Index] Đã lưu {len(vectors)} vector (1024 chiều) vào Cosine HNSW Vector Store.")

# 2. Lập chỉ mục Sparse Okapi BM25
pipeline.bm25_search.index(all_chunks)
pipeline.indexed_chunks = all_chunks
print(f"   [Sparse Index] Đã lập chỉ mục Okapi BM25 trên {len(all_chunks)} chunks.")

# Preview 1 chunk tiêu biểu
sample_chunk = [c for c in all_chunks if c.get("legal_ref")][0]
print("\n" + "=" * 65)
print("BẢN XEM TRƯỚC (PREVIEW) 1 CHUNK TIÊU BIỂU:")
print("=" * 65)
print(f"Chunk ID: {sample_chunk['chunk_id']}")
print(f"Nguồn: {sample_chunk['source']}")
print(f"Breadcrumbs: {sample_chunk['breadcrumb']}")
print(f"Thẻ pháp lý: {sample_chunk['legal_ref']}")
print(f"Nội dung:\n{sample_chunk['content'][:350]}...")


  KHỞI TẠO HỆ THỐNG AI BANKING RAG ASSISTANT (VIETCOMBANK DUAL-INDEX)
[1/6] Khởi tạo Breadcrumbs Chunker (Bảo toàn phân cấp & Thẻ pháp lý)...
[2/6] Khởi tạo Dense Embedding Model 1024 chiều (multilingual-e5-large)...
[3/6] Khởi tạo Dense Vector Store (Cosine HNSW 1024 chiều)...
[4/6] Khởi tạo Sparse Keyword Search (Okapi BM25 k1=1.5, b=0.75)...
[5/6] Khởi tạo Query Parser, Hybrid Retriever (RRF k=60) & Reranker...
[6/6] Khởi tạo Context Synthesizer & LLM Serving (DeepSeek-V4-Flash) kèm Banking Guardrails...
[Hệ thống] Toàn bộ 5 phân hệ đã sẵn sàng hoạt động!

-> Đã trích xuất tổng cộng: 130 chunks có cấu trúc Breadcrumbs.
-> Đang sinh vector nhúng 1024 chiều (multilingual-e5-large) cho 130 chunks...


   [Dense Index] Đã lưu 130 vector (1024 chiều) vào Cosine HNSW Vector Store.
   [Sparse Index] Đã lập chỉ mục Okapi BM25 trên 130 chunks.

BẢN XEM TRƯỚC (PREVIEW) 1 CHUNK TIÊU BIỂU:
Chunk ID: phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc_bc_1
Nguồn: phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md
Breadcrumbs: ĐIỀU KIỆN ĐIỀU KHOẢN MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC VIETCOMBANK (MẪU SỐ 02) > Điều 1. Giải thích từ ngữ:
Thẻ pháp lý: [Điều 1, Khoản 1]
Nội dung:
[Breadcrumbs: ĐIỀU KIỆN ĐIỀU KHOẢN MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC VIETCOMBANK (MẪU SỐ 02) > Điều 1. Giải thích từ ngữ:]\n[Pháp lý: [Điều 1, Khoản 1]]
1. Khách hàng (KH): là cá nhân mở tài khoản thanh toán Vietcombank (VCB) trên kênh trực tuyến.  
2. Tài khoản thanh toán (TK) trực tuyến (TK trực tuyến): là TK tiền gửi không kỳ hạ...


## BƯỚC 1: QUERY PARSER (CHUẨN HÓA TỪ VIẾT TẮT & TRÍCH XUẤT TỪ KHÓA)

Phân hệ **Query Parser** thực hiện:
1. **Chuẩn hóa từ viết tắt nghiệp vụ**:
   - `eKYC` $\rightarrow$ `định danh điện tử (eKYC)`
   - `TKTT` $\rightarrow$ `tài khoản thanh toán (TKTT)`
   - `Smart OTP` $\rightarrow$ `phương thức xác thực Smart OTP`
   - `VCB Digibank` $\rightarrow$ `ngân hàng số VCB Digibank (Vietcombank Digibank)`
   - `KHCN` $\rightarrow$ `khách hàng cá nhân (KHCN)`
   - `CCCD` $\rightarrow$ `căn cước công dân gắn chip (CCCD)`
2. **Nhận diện thực thể pháp lý**: Trích xuất `Điều X`, `Khoản Y` nếu có trong câu hỏi.
3. **Trích xuất từ khóa (Keywords)**: Tách các thuật ngữ trọng tâm phục vụ truy vấn BM25.


In [3]:
test_queries = [
    "Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?",
    "Theo Điều 1 và Điều 2, khách hàng mở TKTT qua eKYC có quyền và nghĩa vụ gì?",
    "Phí duy trì tài khoản thanh toán và phí sử dụng dịch vụ VCB Digibank mỗi tháng là bao nhiêu?"
]

print("=" * 80)
print("KIỂM THỬ BƯỚC 1: QUERY PARSER TRÊN CÁC CÂU HỎI THỰC TẾ")
print("=" * 80)

parsed_results = []
for idx, q in enumerate(test_queries, start=1):
    parsed = pipeline.step1_query_parser(q)
    parsed_results.append(parsed)
    print(f"\n[CÂU HỎI {idx}]")
    print(f"  * Câu gốc:        {parsed.raw_query}")
    print(f"  * Câu chuẩn hóa:  {parsed.normalized_query}")
    print(f"  * Từ viết tắt:    {parsed.detected_acronyms}")
    print(f"  * Thực thể pháp lý: {parsed.legal_references}")
    print(f"  * Từ khóa BM25:   {parsed.keywords[:8]}")


KIỂM THỬ BƯỚC 1: QUERY PARSER TRÊN CÁC CÂU HỎI THỰC TẾ

[CÂU HỎI 1]
  * Câu gốc:        Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?
  * Câu chuẩn hóa:  Hạn mức giao dịch và điều kiện mở tài khoản thanh toán (TKTT) trực tuyến qua định danh điện tử (eKYC) trên app ngân hàng số VCB Digibank (Vietcombank Digibank) xác thực phương thức xác thực Smart OTP là gì?
  * Từ viết tắt:    [('VCB Digibank', 'ngân hàng số VCB Digibank (Vietcombank Digibank)'), ('Smart OTP', 'phương thức xác thực Smart OTP'), ('eKYC', 'định danh điện tử (eKYC)'), ('TKTT', 'tài khoản thanh toán (TKTT)')]
  * Thực thể pháp lý: []
  * Từ khóa BM25:   ['ekyc', 'tktt', 'smart otp', 'vcb digibank', 'hạn', 'mức', 'giao', 'dịch']

[CÂU HỎI 2]
  * Câu gốc:        Theo Điều 1 và Điều 2, khách hàng mở TKTT qua eKYC có quyền và nghĩa vụ gì?
  * Câu chuẩn hóa:  Theo Điều 1 và Điều 2, khách hàng mở tài khoản thanh toán (TKTT) qua định danh điện tử (eKYC) có quyền và ngh

## BƯỚC 2: HYBRID RETRIEVAL (TRUY VẤN SONG SONG & RRF k=60)

Truy vấn đồng thời hai luồng:
1. **Luồng Dense**: Embedding câu truy vấn bằng vector 1024-d, tìm kiếm trên Cosine HNSW Vector Store.
2. **Luồng Sparse**: Tìm kiếm khớp từ khóa chính xác bằng Okapi BM25.
3. **Reciprocal Rank Fusion (RRF $k=60$)**:
$$RRF(d) = \frac{0.5}{60 + \text{rank}_{\text{dense}}(d)} + \frac{0.5}{60 + \text{rank}_{\text{bm25}}(d)}$$

Thuật toán kết hợp cả hai thứ hạng và trích xuất đúng **Top-5 ứng viên** tiềm năng nhất.


In [4]:
primary_query = parsed_results[0]
candidates_top5 = pipeline.step2_hybrid_retrieval(primary_query, top_k=5)

# Trình bày kết quả Top-5 dưới dạng DataFrame trực quan
df_top5 = []
for c in candidates_top5:
    chunk = c["chunk"]
    df_top5.append({
        "Top (RRF)": c["candidate_rank"],
        "RRF Score": c["rrf_score"],
        "Dense Rank": c["dense_rank"] if c["dense_rank"] else "-",
        "Dense Score": c["dense_score"],
        "BM25 Rank": c["bm25_rank"] if c["bm25_rank"] else "-",
        "BM25 Score": c["bm25_score"],
        "Nguồn tài liệu": chunk.get("source"),
        "Căn cứ pháp lý": chunk.get("legal_ref", "Quy chế"),
        "Đường dẫn (Breadcrumbs)": chunk.get("breadcrumb", "")[:45] + "..."
    })

df_display_top5 = pd.DataFrame(df_top5)
print("KẾT QUẢ BƯỚC 2: TOP-5 ỨNG VIÊN HỢP NHẤT BẰNG THUẬT TOÁN RRF (k=60)")
display(df_display_top5)


KẾT QUẢ BƯỚC 2: TOP-5 ỨNG VIÊN HỢP NHẤT BẰNG THUẬT TOÁN RRF (k=60)


,Top (RRF),RRF Score,Dense Rank,Dense Score,BM25 Rank,BM25 Score,Nguồn tài liệu,Căn cứ pháp lý,Đường dẫn (Breadcrumbs)
0,1,0.016009,1,0.8985,4,35.7914,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,,BỘ CÂU HỎI THƯỜNG GẶP (FAQ): NGÂN HÀNG SỐ VIE...
1,2,0.015889,5,0.8868,1,42.3580,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,,BỘ CÂU HỎI THƯỜNG GẶP (FAQ): NGÂN HÀNG SỐ VIE...
2,3,0.015877,4,0.8894,2,40.5025,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,,BỘ CÂU HỎI THƯỜNG GẶP (FAQ): NGÂN HÀNG SỐ VIE...
3,4,0.015527,2,0.8976,7,30.3634,phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_tru...,"[Điều 1, Khoản 1]",ĐIỀU KIỆN ĐIỀU KHOẢN MỞ VÀ SỬ DỤNG TÀI KHOẢN ...
4,5,0.015289,8,0.8847,3,36.7271,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,,BỘ CÂU HỎI THƯỜNG GẶP (FAQ): NGÂN HÀNG SỐ VIE...


## BƯỚC 3: RERANKING (SO KHỚP CÂU HỎI CHUẨN & ĐỘ TRÙNG LẶP TỪ VỰNG)

Để tinh lọc từ Top-5 ứng viên xuống **Top-3 chunks** chuẩn xác nhất, thuật toán đánh giá đa tiêu chí:
1. **So khớp với câu hỏi chuẩn (Canonical Questions Matching)**:
   - Các chunk tài liệu ngân hàng chứa các câu hỏi thường gặp hoặc tiêu đề mục chuẩn: $\text{Score}_{\text{canonical}} = \max_{q_c} \text{Similarity}(Q, q_c)$.
2. **Độ trùng lặp từ vựng (Lexical Overlap & Keyword Containment)**:
   - Tỷ lệ từ khóa và token trong câu hỏi xuất hiện trong nội dung chunk: $\text{Score}_{\text{lexical}} = \frac{|Tokens(Q) \cap Tokens(Chunk)|}{|Tokens(Q)|}$.
3. **Điểm tổng hợp Rerank**:
$$\text{FinalRerankScore} = 0.40 \times \text{Score}_{\text{canonical}} + 0.35 \times \text{Score}_{\text{lexical}} + 0.25 \times \text{NormalizedRRF}$$


In [5]:
top3_chunks = pipeline.step3_reranking(primary_query, candidates_top5, top_k=3)

df_top3 = []
for c in top3_chunks:
    chunk = c["chunk"]
    df_top3.append({
        "Rerank Top": c["rerank_rank"],
        "Hybrid Top Gốc": c["original_hybrid_rank"],
        "Final Rerank Score": c["final_rerank_score"],
        "Canonical Match": c["canonical_matching_score"],
        "Lexical Overlap": c["lexical_overlap_score"],
        "RRF Score": c["rrf_score"],
        "Nguồn tài liệu": chunk.get("source"),
        "Căn cứ pháp lý": chunk.get("legal_ref", "Quy chế")
    })

df_display_top3 = pd.DataFrame(df_top3)
print("KẾT QUẢ BƯỚC 3: TINH LỌC RA TOP-3 CHUNKS CHÍNH XÁC NHẤT")
display(df_display_top3)


KẾT QUẢ BƯỚC 3: TINH LỌC RA TOP-3 CHUNKS CHÍNH XÁC NHẤT


,Rerank Top,Hybrid Top Gốc,Final Rerank Score,Canonical Match,Lexical Overlap,RRF Score,Nguồn tài liệu,Căn cứ pháp lý
0,1,4,0.6456,0.1000,0.7941,0.015527,phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_tru...,"[Điều 1, Khoản 1]"
1,2,1,0.6268,0.5000,0.5386,0.016009,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,
2,3,3,0.6016,0.5077,0.5229,0.015877,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,


## BƯỚC 4: CONTEXT SYNTHESIZER (ĐÓNG GÓI MARKDOWN, GẮN THẺ PHÁP LÝ & KIỂM SOÁT TOKEN)

Phân hệ **Context Synthesizer** đảm nhiệm:
1. **Đóng gói Markdown chuyên nghiệp**: Định dạng từng đoạn trích với nguồn, đường dẫn breadcrumbs, và nội dung xác thực.
2. **Gắn thẻ căn cứ pháp lý**: Tự động nhận diện và gắn thẻ chuẩn `[Điều X, Khoản Y]` làm bằng chứng trích dẫn.
3. **Kiểm soát giới hạn Token (Token Budget Control)**: Giới hạn tối đa **2000 tokens**, tự động cắt tỉa an toàn bảo đảm không làm rách điều khoản pháp lý và chống tràn context window.


In [6]:
synthesized_context = pipeline.step4_context_synthesizer(top3_chunks, max_tokens=2000)

print("=" * 75)
print("KẾT QUẢ BƯỚC 4: THÔNG SỐ NGỮ CẢNH ĐÃ ĐÓNG GÓI")
print("=" * 75)
print(f"Tổng số token ước tính:  {synthesized_context.total_tokens} (Ngân sách tối đa: 2000)")
print(f"Căn cứ pháp lý trích dẫn: {synthesized_context.legal_citations}")
print(f"Tài liệu nguồn sử dụng:   {synthesized_context.sources_used}")
print(f"Số chunks đã gói:        {synthesized_context.chunks_included} / 3\n")

print("--- NỘI DUNG MARKDOWN CONTEXT SẼ GỬI SANG LLM ---")
print(synthesized_context.markdown_context[:850] + "\n...\n[Đoạn trích tiếp tục bên dưới]")


KẾT QUẢ BƯỚC 4: THÔNG SỐ NGỮ CẢNH ĐÃ ĐÓNG GÓI
Tổng số token ước tính:  1503 (Ngân sách tối đa: 2000)
Căn cứ pháp lý trích dẫn: ['[Điều 1, Khoản 1]', '[Điều 1]']
Tài liệu nguồn sử dụng:   ['phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md', 'faq__ngan_hang_so_vcb_digibank_va_smart_otp.md']
Số chunks đã gói:        3 / 3

--- NỘI DUNG MARKDOWN CONTEXT SẼ GỬI SANG LLM ---
### [TRÍCH DẪN 1] Nguồn: phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md
- **Tham chiếu pháp lý:** `[Điều 1, Khoản 1], [Điều 1]`
- **Đường dẫn phân cấp (Breadcrumbs):** `ĐIỀU KIỆN ĐIỀU KHOẢN MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC VIETCOMBANK (MẪU SỐ 02) > Điều 1. Giải thích từ ngữ:`
- **Nội dung xác thực:**
1. Khách hàng (KH): là cá nhân mở tài khoản thanh toán Vietcombank (VCB) trên kênh trực tuyến.  
2. Tài khoản thanh toán (TK) trực tuyến (TK trực tuyến): là TK tiền gửi không kỳ hạn đồng Việt Nam (VND) của 
KH mở trên kênh ngân hàng điện tử (NHĐT) của VCB hoặc kênh NHĐT khác theo

## BƯỚC 5: LLM SERVING (HỆ THỐNG SYSTEM GUARDRAILS NGÂN HÀNG & SUY LUẬN)

### 4 Nguyên tắc System Guardrails cốt lõi:
1. **Zero-Hallucination & Chính xác số liệu**: Tuyệt đối không bịa đặt số liệu (lãi suất %, hạn mức VND, biểu phí). Mọi số liệu phải có nguyên văn từ tài liệu.
2. **Dẫn chứng pháp lý bắt buộc**: Trích dẫn cụ thể `[Điều X, Khoản Y]` cho mỗi điều kiện nghiệp vụ.
3. **An toàn bảo mật**: Nghiêm cấm yêu cầu và cảnh báo khách hàng không cung cấp mã OTP, PIN, mật khẩu, CVV.
4. **Minh bạch phạm vi**: Nếu tài liệu không đề cập, phải thừa nhận rõ ràng và hướng dẫn gọi Hotline Vietcombank 1900 54 54 13.


In [7]:
serving_result = pipeline.step5_llm_serving(
    query=primary_query.raw_query,
    synthesized_context=synthesized_context
)

print("=" * 75)
print(f"CÂU TRẢ LỜI TỪ MÔ HÌNH: {serving_result['model_served']}")
print("=" * 75)
print(serving_result["answer"])


CÂU TRẢ LỜI TỪ MÔ HÌNH: DeepSeek-V4-Flash
Chúng ta cần trả lời câu hỏi: "Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?"

Dựa trên ngữ cảnh cung cấp, có hai phần: điều kiện mở tài khoản trực tuyến eKYC và hạn mức giao dịch. Cần trích dẫn điều khoản cụ thể. Trích dẫn 1 nói về định nghĩa và hạn mức: "Đối với các KH đã mở thành công TK trực tuyến và chưa xác thực trực tiếp tại ĐGD của VCB, tổng hạn mức ghi nợ TK tối đa là 400 triệu VND/tháng/KH (trừ ghi nợ để (i) mở/nộp thêm tiền vào tài khoản tiền gửi có kỳ hạn, (ii) đăng ký tiết kiệm tự động trên kênh NHĐT và (iii) giao dịch thẻ)." Điều này nằm trong Điều 1, Khoản 12? Thực tế trong văn bản, phần "Hạn mức giao dịch..." là mục 12 của Điều 1. Cần trích dẫn [Điều 1, Khoản 12]. Nhưng trong trích dẫn, breadcrumbs chỉ đến Điều 1. Giải thích từ ngữ, và nội dung có 12 mục. Vậy có thể trích dẫn [Điều 1, Khoản 12].

Điều kiện mở TKTT trực tuyến qua eKYC: Trong Trích dẫn 1, Điều 1, Khoản 3

## THỬ NGHIỆM TOÀN TRÌNH TRÊN CÁC KỊCH BẢN NGHIỆP VỤ THỰC TẾ & ĐO ĐỘ TRỄ (LATENCY BENCHMARK)

Kiểm thử pipeline tự động trên 3 kịch bản ngân hàng thực tế:
1. **Kịch bản 1**: Điều kiện pháp lý mở tài khoản eKYC và Hạn mức giao dịch.
2. **Kịch bản 2**: Phương thức xác thực Smart OTP trên ứng dụng VCB Digibank.
3. **Kịch bản 3**: Biểu phí duy trì tài khoản thanh toán và quy định số dư tối thiểu.


In [8]:
benchmark_queries = [
    "Khách hàng cá nhân mở TKTT trực tuyến qua eKYC chưa xác thực tại quầy thì hạn mức giao dịch tối đa là bao nhiêu theo Điều 1?",
    "Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?",
    "Phí mở tài khoản thanh toán và số dư tối thiểu yêu cầu tại Vietcombank là bao nhiêu?"
]

benchmark_records = []

for idx, q_text in enumerate(benchmark_queries, start=1):
    print(f"\n{'='*70}")
    print(f"THỰC THI TOÀN TRÌNH KỊCH BẢN {idx}: '{q_text}'")
    print(f"{'='*70}")
    
    res = pipeline.run_pipeline(q_text)
    
    print(f"-> Căn cứ pháp lý phát hiện: {res['legal_citations']}")
    print(f"-> Nguồn tài liệu sử dụng:   {res['sources_used']}")
    print(f"-> Độ trễ toàn trình:       {res['latency']['total_pipeline_ms']} ms")
    print("\n[CÂU TRẢ LỜI CỦA TRỢ LÝ]:")
    print(res["answer"])
    
    lat = res["latency"]
    benchmark_records.append({
        "Kịch bản": f"Kịch bản {idx}",
        "Câu hỏi": q_text[:40] + "...",
        "Query Parser (ms)": lat["step1_query_parser_ms"],
        "Hybrid RRF (ms)": lat["step2_hybrid_retrieval_ms"],
        "Reranking (ms)": lat["step3_reranking_ms"],
        "Synthesizer (ms)": lat["step4_context_synthesizer_ms"],
        "LLM Serving (ms)": lat["step5_llm_serving_ms"],
        "Tổng thời gian (ms)": lat["total_pipeline_ms"]
    })

df_benchmark = pd.DataFrame(benchmark_records)
print("\n" + "=" * 70)
print("BẢNG TỔNG HỢP HIỆU NĂNG & ĐỘ TRỄ HỆ THỐNG (LATENCY BENCHMARK)")
print("=" * 70)
display(df_benchmark)



THỰC THI TOÀN TRÌNH KỊCH BẢN 1: 'Khách hàng cá nhân mở TKTT trực tuyến qua eKYC chưa xác thực tại quầy thì hạn mức giao dịch tối đa là bao nhiêu theo Điều 1?'

[QUERY] 'Khách hàng cá nhân mở TKTT trực tuyến qua eKYC chưa xác thực tại quầy thì hạn mức giao dịch tối đa là bao nhiêu theo Điều 1?'


-> Căn cứ pháp lý phát hiện: ['[Điều 1, Khoản 1]', '[Điều 1]', '[Điều 2, Khoản 6]', '[Điều 2]', '[Điều 4, Khoản 8]', '[Điều 4]']
-> Nguồn tài liệu sử dụng:   ['phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md']
-> Độ trễ toàn trình:       14284.6 ms

[CÂU TRẢ LỜI CỦA TRỢ LÝ]:
Kính thưa Quý khách,

Theo quy định tại **[Điều 1, Khoản 12]** trong Điều kiện, Điều khoản mở và sử dụng Tài khoản thanh toán trực tuyến eKYC của Vietcombank:

- Đối với khách hàng cá nhân đã mở thành công tài khoản thanh toán trực tuyến qua eKYC **nhưng chưa xác thực trực tiếp tại Điểm giao dịch (ĐGD)** của Vietcombank, **tổng hạn mức ghi nợ tài khoản tối đa là 400 triệu đồng/tháng/khách hàng**.

Quý khách lưu ý, hạn mức trên **không áp dụng** đối với các khoản ghi nợ để:
- (i) Mở/nộp thêm tiền vào tài khoản tiền gửi có kỳ hạn;
- (ii) Đăng ký tiết kiệm tự động trên kênh ngân hàng điện tử;
- (iii) Giao dịch thẻ.

Sau khi Quý khách hoàn tất xác thực trực tiếp tại ĐGD của Vietcombank, tài khoản sẽ được 

-> Căn cứ pháp lý phát hiện: []
-> Nguồn tài liệu sử dụng:   ['faq__ngan_hang_so_vcb_digibank_va_smart_otp.md']
-> Độ trễ toàn trình:       10770.4 ms

[CÂU TRẢ LỜI CỦA TRỢ LÝ]:
Chúng ta cần trả lời câu hỏi: "Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?" Dựa trên ngữ cảnh cung cấp.

Chúng ta có các trích dẫn:

- Trích dẫn 1: FAQ câu 14: Sau khi tải app VCB-Smart OTP, nhập mã kích hoạt ứng dụng được Vietcombank gửi vào Số điện thoại đăng ký VCB-Smart OTP, Quý khách cần lưu ý thực hiện 2 giao dịch tài chính (chuyển tiền, thanh toán, nạp tiền, tiết kiệm…) xác thực bằng SMS OTP trên VCB Digibank để có thể bắt đầu xác thực giao dịch bằng VCB-Smart OTP. Đây là điều kiện kích hoạt.

- Trích dẫn 2: FAQ câu 2: Với các phương thức xác thực giao dịch Vietcombank đang cung cấp (SMS OTP, VCB-Smart OTP), Quý khách hàng đều có thể thực hiện được giao dịch trên VCB Digibank khi ra nước ngoài. Đối với hình thức nhận OTP qua SMS, cần đăng ký chuyển vùng

-> Căn cứ pháp lý phát hiện: ['[Điều 1, Khoản 1]', '[Điều 1]', '[Điều 2, Khoản 3]', '[Điều 2]']
-> Nguồn tài liệu sử dụng:   ['phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md', 'phap_ly__dieu_kien_dieu_khoan_mo_va_su_dung_tai_khoan_khcn.md']
-> Độ trễ toàn trình:       11095.9 ms

[CÂU TRẢ LỜI CỦA TRỢ LÝ]:
Kính thưa Quý khách,

Vietcombank xin ghi nhận câu hỏi của Quý khách về **phí mở tài khoản thanh toán và số dư tối thiểu yêu cầu**. 

Sau khi rà soát các tài liệu nội bộ và căn cứ pháp lý hiện đang được cung cấp trong hệ thống, chúng tôi nhận thấy các văn bản này chỉ tập trung quy định về:

- **Giải thích từ ngữ** liên quan đến tài khoản thanh toán, tài khoản trực tuyến eKYC [Điều 1, Khoản 1].
- **Quyền và nghĩa vụ của khách hàng** khi sử dụng tài khoản [Điều 2, Khoản 3].
- **Hạn mức giao dịch** tối đa đối với tài khoản trực tuyến chưa xác thực trực tiếp (400 triệu VND/tháng) [Điều 1, Khoản 12 – Trích dẫn 2].

Tuy nhiên, **các tài liệu được cung cấp hiện không bao gồm n

,Kịch bản,Câu hỏi,Query Parser (ms),Hybrid RRF (ms),Reranking (ms),Synthesizer (ms),LLM Serving (ms),Tổng thời gian (ms)
0,Kịch bản 1,Khách hàng cá nhân mở TKTT trực tuyến qu...,0.2,214.8,197.1,0.6,13871.9,14284.6
1,Kịch bản 2,Smart OTP là gì và điều kiện để kích hoạ...,0.2,302.8,271.6,0.2,10195.6,10770.4
2,Kịch bản 3,Phí mở tài khoản thanh toán và số dư tối...,0.3,213.5,225.4,0.7,10655.8,11095.9


## KẾT LUẬN & ĐÁNH GIÁ HỆ THỐNG

1. **Pha Ingestion & Dual-Indexing**:
   - Hoàn thành xuất sắc việc phân đoạn bảo toàn theo dạng Breadcrumbs.
   - Dual-Indexing kết hợp trọn vẹn sức mạnh của **Cosine HNSW 1024 chiều** (nắm bắt ngữ nghĩa) và **Okapi BM25** (bắt chính xác mã, biểu phí và từ khóa kỹ thuật).
2. **Bước 1 (Query Parser)**:
   - Chuẩn hóa thành công 100% các từ viết tắt tài chính (`eKYC`, `TKTT`, `Smart OTP`, `VCB Digibank`, `KHCN`, `CCCD`).
3. **Bước 2 (Hybrid Retrieval)**:
   - Thuật toán **Reciprocal Rank Fusion (RRF $k=60$)** kết hợp chuẩn xác hai luồng truy vấn, trích xuất chính xác **Top-5 ứng viên**.
4. **Bước 3 (Reranking)**:
   - Đánh giá trên 2 tiêu chí cốt lõi: So khớp câu hỏi chuẩn (`Canonical Questions`) và Độ trùng lặp từ vựng (`Lexical Overlap`), tinh lọc thành công ra **Top-3 chunks**.
5. **Bước 4 (Context Synthesizer)**:
   - Đóng gói Markdown chuyên nghiệp, tự động gắn thẻ pháp lý `[Điều X, Khoản Y]` và kiểm soát an toàn ngân sách **2000 tokens**.
6. **Bước 5 (LLM Serving)**:
   - Triển khai **Banking System Guardrails** nghiêm ngặt: loại bỏ triệt để ảo giác số liệu tài chính, bảo mật thông tin OTP/PIN, giải đáp minh bạch và chuẩn mực ngôn phong ngân hàng.
